# NFL Week 1 Analysis — Data Preparation

This notebook loads, cleans, validates, and prepares the NFL game-level and team-game-level data used by the analysis notebooks.

**Source data:** `nflreadpy` team statistics and the Kaggle NFL scores/betting dataset.

In [1]:
import nflreadpy as nfl
import pandas as pd
import numpy as np
import pyarrow
import kagglehub
from kagglehub import KaggleDatasetAdapter


c:\Users\natel\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load NFL team statistics

In [2]:
team_stats_2024 = nfl.load_team_stats(
    seasons=2024,
    summary_level="week"
)

print(team_stats_2024)
print(team_stats_2024.columns)
print(team_stats_2024.shape)

shape: (570, 138)
┌────────┬──────┬──────┬─────────────┬───┬─────────────┬──────────────┬──────────────┬─────────────┐
│ season ┆ week ┆ team ┆ season_type ┆ … ┆ pt_returned ┆ pt_return_ya ┆ pt_return_td ┆ pt_net_yard │
│ ---    ┆ ---  ┆ ---  ┆ ---         ┆   ┆ ---         ┆ rds          ┆ s            ┆ s           │
│ i32    ┆ i32  ┆ str  ┆ str         ┆   ┆ i32         ┆ ---          ┆ ---          ┆ ---         │
│        ┆      ┆      ┆             ┆   ┆             ┆ i32          ┆ i32          ┆ i32         │
╞════════╪══════╪══════╪═════════════╪═══╪═════════════╪══════════════╪══════════════╪═════════════╡
│ 2024   ┆ 1    ┆ ARI  ┆ REG         ┆ … ┆ 1           ┆ 7            ┆ 0            ┆ 71          │
│ 2024   ┆ 1    ┆ ATL  ┆ REG         ┆ … ┆ 4           ┆ 47           ┆ 0            ┆ 163         │
│ 2024   ┆ 1    ┆ BAL  ┆ REG         ┆ … ┆ 0           ┆ 0            ┆ 0            ┆ 75          │
│ 2024   ┆ 1    ┆ BUF  ┆ REG         ┆ … ┆ 1           ┆ 6            ┆ 0

In [3]:
print(team_stats_2024["game_id"])
print(team_stats_2024["season_type"])

shape: (570,)
Series: 'game_id' [str]
[
	"2024_01_ARI_BUF"
	"2024_01_PIT_ATL"
	"2024_01_BAL_KC"
	"2024_01_ARI_BUF"
	"2024_01_CAR_NO"
	…
	"2024_21_BUF_KC"
	"2024_21_WAS_PHI"
	"2024_21_WAS_PHI"
	"2024_22_KC_PHI"
	"2024_22_KC_PHI"
]
shape: (570,)
Series: 'season_type' [str]
[
	"REG"
	"REG"
	"REG"
	"REG"
	"REG"
	…
	"POST"
	"POST"
	"POST"
	"POST"
	"POST"
]


In [4]:

team_stats_full = nfl.load_team_stats(
    seasons=list(range(2006, 2025)),
    summary_level="week"
)

In [5]:
keep_col = [
    # Identifiers
    'season',
    'week',
    'team',
    'season_type',
    'game_id',
    'opponent_team',
    # Passing
    'completions',
    'attempts',
    'passing_yards',
    'passing_tds',
    'passing_interceptions',
    'sacks_suffered',
    'passing_air_yards',
    'passing_yards_after_catch',
    'passing_first_downs',
    'passing_epa',
    'passing_cpoe',
    # Rushing
    'carries',
    'rushing_yards',
    'rushing_tds',
    'rushing_first_downs',
    'rushing_epa',
    # Defense
    'def_tackles_solo',
    'def_tackles_with_assist',
    'def_tackle_assists',
    'def_tackles_for_loss',
    'def_fumbles_forced',
    'def_sacks',
    'def_sack_yards',
    'def_qb_hits',
    'def_interceptions',
    'def_pass_defended',
    'def_tds',
    'def_fumbles',
    'def_safeties',
    # Turnovers / penalties
    'fumbles_total',
    'fumbles_lost_total',
    'penalties',
    'penalty_yards',
    # Special teams
    'fg_made',
    'fg_att',
    'fg_missed',
    'fg_blocked',
    'fg_long',
    'fg_pct',
    'punt_returns',
    'punt_return_yards',
    'kickoff_returns',
    'kickoff_return_yards'
]

In [6]:
team_stats_w_post = team_stats_full.select(keep_col).to_pandas()

In [7]:
team_stats = team_stats_w_post[
    team_stats_w_post["season_type"] == "REG"
].copy()

In [8]:
print("Shape:", team_stats.shape)

print("\nSeasons:")
print(team_stats["season"].unique())

print("\nGames by season:")
print(
    team_stats
    .groupby("season")["game_id"]
    .nunique()
)

print("\nRows / game:")
print(
    team_stats
    .groupby("game_id")
    .size()
    .value_counts()
    .sort_index()
)

print("\nMissing vals:")
print(
    team_stats
    .isna()
    .sum()
    .sort_values(ascending=False)
)

Shape: (9854, 49)

Seasons:
[2006 2007 2008 2009 2010 2011 2012 2013 2014 2015 2016 2017 2018 2019
 2020 2021 2022 2023 2024]

Games by season:
season
2006    256
2007    256
2008    256
2009    256
2010    256
2011    256
2012    256
2013    256
2014    256
2015    256
2016    256
2017    256
2018    256
2019    256
2020    256
2021    272
2022    271
2023    272
2024    272
Name: game_id, dtype: int64

Rows / game:
2    4927
Name: count, dtype: int64

Missing vals:
fg_long                      1812
fg_pct                       1261
passing_cpoe                   11
season                          0
week                            0
opponent_team                   0
completions                     0
season_type                     0
team                            0
passing_yards                   0
passing_tds                     0
sacks_suffered                  0
passing_interceptions           0
passing_air_yards               0
passing_yards_after_catch       0
attempts          

In [9]:
team_stats[
    team_stats["passing_cpoe"].isna()
][
    [
        "season",
        "week",
        "team",
        "opponent_team",
        "game_id",
        "completions",
        "attempts",
        "passing_yards",
        "passing_cpoe"
    ]
]

,season,week,team,opponent_team,game_id,completions,attempts,passing_yards,passing_cpoe
106,2006,4,KC,SF,2006_04_SF_KC,18,23,208,NaN
117,2006,4,SF,KC,2006_04_SF_KC,13,25,92,NaN
187,2006,7,KC,LAC,2006_07_SD_KC,15,27,232,NaN
188,2006,7,LAC,KC,2006_07_SD_KC,26,44,267,NaN
213,2006,8,KC,SEA,2006_08_SEA_KC,17,25,312,NaN
224,2006,8,SEA,KC,2006_08_SEA_KC,15,30,198,NaN
306,2006,11,LV,KC,2006_11_OAK_KC,15,25,194,NaN
386,2006,14,BAL,KC,2006_14_BAL_KC,21,27,283,NaN
399,2006,14,KC,BAL,2006_14_BAL_KC,15,27,178,NaN
494,2006,17,JAX,KC,2006_17_JAX_KC,23,40,306,NaN


## 2. Load and filter betting data

In [10]:
betting_data = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "tobycrabtree/nfl-scores-and-betting-data",
    "spreadspoke_scores.csv"
)

print("Shape:", betting_data.shape)
print("\nColumns:")
print(betting_data.columns.tolist())

print("\nFirst 5 rows:")
print(betting_data.head())

Shape: (14371, 17)

Columns:
['schedule_date', 'schedule_season', 'schedule_week', 'schedule_playoff', 'team_home', 'score_home', 'score_away', 'team_away', 'team_favorite_id', 'spread_favorite', 'over_under_line', 'stadium', 'stadium_neutral', 'weather_temperature', 'weather_wind_mph', 'weather_humidity', 'weather_detail']

First 5 rows:
  schedule_date  schedule_season schedule_week  schedule_playoff  \
0      9/2/1966             1966             1             False   
1      9/3/1966             1966             1             False   
2      9/4/1966             1966             1             False   
3      9/9/1966             1966             2             False   
4     9/10/1966             1966             1             False   

            team_home  score_home  score_away        team_away  \
0      Miami Dolphins          14          23  Oakland Raiders   
1      Houston Oilers          45           7   Denver Broncos   
2  San Diego Chargers          27           7    Buf

In [11]:
betting = betting_data[
    (betting_data["schedule_season"] >= 2006) &
    (betting_data["schedule_season"] <= 2024) &
    (betting_data["schedule_playoff"] == False)
].copy()

print("data shape:", betting.shape)

print("\nGames by season:")
print(
    betting
    .groupby("schedule_season")
    .size()
)

data shape: (4927, 17)

Games by season:
schedule_season
2006    256
2007    256
2008    256
2009    256
2010    256
2011    256
2012    256
2013    256
2014    256
2015    256
2016    256
2017    256
2018    256
2019    256
2020    256
2021    272
2022    271
2023    272
2024    272
dtype: int64


## 3. Standardize team names

In [12]:
print("Teams in team_stats:")
print(sorted(team_stats["team"].unique()))

print("\nHome teams in betting data:")
print(sorted(betting["team_home"].unique()))

print("\nAway teams in betting data:")
print(sorted(betting["team_away"].unique()))

Teams in team_stats:
['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAX', 'KC', 'LA', 'LAC', 'LV', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'PHI', 'PIT', 'SEA', 'SF', 'TB', 'TEN', 'WAS']

Home teams in betting data:
['Arizona Cardinals', 'Atlanta Falcons', 'Baltimore Ravens', 'Buffalo Bills', 'Carolina Panthers', 'Chicago Bears', 'Cincinnati Bengals', 'Cleveland Browns', 'Dallas Cowboys', 'Denver Broncos', 'Detroit Lions', 'Green Bay Packers', 'Houston Texans', 'Indianapolis Colts', 'Jacksonville Jaguars', 'Kansas City Chiefs', 'Las Vegas Raiders', 'Los Angeles Chargers', 'Los Angeles Rams', 'Miami Dolphins', 'Minnesota Vikings', 'New England Patriots', 'New Orleans Saints', 'New York Giants', 'New York Jets', 'Oakland Raiders', 'Philadelphia Eagles', 'Pittsburgh Steelers', 'San Diego Chargers', 'San Francisco 49ers', 'Seattle Seahawks', 'St. Louis Rams', 'Tampa Bay Buccaneers', 'Tennessee Titans', 'Washington Commanders', 'Washington 

In [13]:

team_name_map = {
    "Arizona Cardinals": "ARI",
    "Atlanta Falcons": "ATL",
    "Baltimore Ravens": "BAL",
    "Buffalo Bills": "BUF",
    "Carolina Panthers": "CAR",
    "Chicago Bears": "CHI",
    "Cincinnati Bengals": "CIN",
    "Cleveland Browns": "CLE",
    "Dallas Cowboys": "DAL",
    "Denver Broncos": "DEN",
    "Detroit Lions": "DET",
    "Green Bay Packers": "GB",
    "Houston Texans": "HOU",
    "Indianapolis Colts": "IND",
    "Jacksonville Jaguars": "JAX",
    "Kansas City Chiefs": "KC",
    "Las Vegas Raiders": "LV",
    "Los Angeles Chargers": "LAC",
    "Los Angeles Rams": "LA",
    "Miami Dolphins": "MIA",
    "Minnesota Vikings": "MIN",
    "New England Patriots": "NE",
    "New Orleans Saints": "NO",
    "New York Giants": "NYG",
    "New York Jets": "NYJ",
    "Philadelphia Eagles": "PHI",
    "Pittsburgh Steelers": "PIT",
    "San Francisco 49ers": "SF",
    "Seattle Seahawks": "SEA",
    "Tampa Bay Buccaneers": "TB",
    "Tennessee Titans": "TEN",
    "Oakland Raiders": "LV",
    "San Diego Chargers": "LAC",
    "St. Louis Rams": "LA",
    "Washington Redskins": "WAS",
    "Washington Football Team": "WAS",
    "Washington Commanders": "WAS"
}
betting["team_home"] = betting["team_home"].map(team_name_map)
betting["team_away"] = betting["team_away"].map(team_name_map)

In [14]:
print("Unmapped home teams:")
print(betting["team_home"].isna().sum())

print("Unmapped away teams:")
print(betting["team_away"].isna().sum())

print(sorted(betting["team_home"].dropna().unique()))

Unmapped home teams:
0
Unmapped away teams:
0
['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAX', 'KC', 'LA', 'LAC', 'LV', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'PHI', 'PIT', 'SEA', 'SF', 'TB', 'TEN', 'WAS']


## 4. Construct and validate game IDs

In [15]:
betting["game_id"] = (
    betting["schedule_season"].astype(str)
    + "_"
    + betting["schedule_week"].astype(str).str.zfill(2)
    + "_"
    + betting["team_away"]
    + "_"
    + betting["team_home"]
)

In [16]:
print("betting games:", betting["game_id"].nunique())
print("team_stats games:", team_stats["game_id"].nunique())
print(
    "Matching games:",
    betting["game_id"].isin(team_stats["game_id"]).sum()
)
print(
    "Team stats games with betting data:",
    team_stats["game_id"].isin(betting["game_id"]).sum()
)

betting games: 4927
team_stats games: 4927
Matching games: 4395
Team stats games with betting data: 8790


In [17]:
unmatched_betting = betting[
    ~betting["game_id"].isin(team_stats["game_id"])
].copy()

print("Unmatched betting games:", len(unmatched_betting))

print(
    unmatched_betting[
        ["schedule_season", "schedule_week",
         "team_away", "team_home", "game_id"]
    ].head(20)
)

Unmatched betting games: 532
      schedule_season schedule_week team_away team_home          game_id
8951             2006             1       DEN        LA   2006_01_DEN_LA
8954             2006             1       LAC        LV   2006_01_LAC_LV
8957             2006             2        LV       BAL   2006_02_LV_BAL
8968             2006             2       TEN       LAC  2006_02_TEN_LAC
8969             2006             2        LA        SF    2006_02_LA_SF
8972             2006             3        LA       ARI   2006_03_LA_ARI
8987             2006             4       LAC       BAL  2006_04_LAC_BAL
8995             2006             4       CLE        LV   2006_04_CLE_LV
8996             2006             4       DET        LA   2006_04_DET_LA
9003             2006             5        LA        GB    2006_05_LA_GB
9011             2006             5       PIT       LAC  2006_05_PIT_LAC
9012             2006             5        LV        SF    2006_05_LV_SF
9017             2006 

In [18]:
unmatched_stats = team_stats[
    ~team_stats["game_id"].isin(betting["game_id"])
].copy()

print("Unmatched team stats rows:", len(unmatched_stats))

print(
    unmatched_stats[
        ["season", "week", "team", "opponent_team", "game_id"]
    ].head(20)
)

Unmatched team stats rows: 1064
     season  week team opponent_team          game_id
9      2006     1  DEN            LA  2006_01_DEN_STL
16     2006     1   LA           DEN  2006_01_DEN_STL
17     2006     1  LAC            LV   2006_01_SD_OAK
18     2006     1   LV           LAC   2006_01_SD_OAK
34     2006     2  BAL            LV  2006_02_OAK_BAL
48     2006     2   LA            SF   2006_02_STL_SF
49     2006     2  LAC           TEN   2006_02_TEN_SD
50     2006     2   LV           BAL  2006_02_OAK_BAL
60     2006     2   SF            LA   2006_02_STL_SF
62     2006     2  TEN           LAC   2006_02_TEN_SD
64     2006     3  ARI            LA  2006_03_STL_ARI
78     2006     3   LA           ARI  2006_03_STL_ARI
94     2006     4  BAL           LAC   2006_04_SD_BAL
99     2006     4  CLE            LV  2006_04_CLE_OAK
101    2006     4  DET            LA  2006_04_DET_STL
107    2006     4   LA           DET  2006_04_DET_STL
108    2006     4  LAC           BAL   2006_04_SD_

In [19]:
game_id_map = {
    "STL": "LA",
    "SD": "LAC",
    "OAK": "LV"
}

team_stats["game_id_normalized"] = (
    team_stats["game_id"]
    .str.split("_")
    .apply(
        lambda x: "_".join([
            x[0],
            x[1],
            game_id_map.get(x[2], x[2]),
            game_id_map.get(x[3], x[3])
        ])
    )
)

In [20]:
print(
    team_stats[
        ["game_id", "game_id_normalized"]
    ].drop_duplicates().head(20)
)

            game_id game_id_normalized
0    2006_01_SF_ARI     2006_01_SF_ARI
1   2006_01_ATL_CAR    2006_01_ATL_CAR
2    2006_01_BAL_TB     2006_01_BAL_TB
3    2006_01_BUF_NE     2006_01_BUF_NE
5    2006_01_CHI_GB     2006_01_CHI_GB
6    2006_01_CIN_KC     2006_01_CIN_KC
7    2006_01_NO_CLE     2006_01_NO_CLE
8   2006_01_DAL_JAX    2006_01_DAL_JAX
9   2006_01_DEN_STL     2006_01_DEN_LA
10  2006_01_SEA_DET    2006_01_SEA_DET
12  2006_01_PHI_HOU    2006_01_PHI_HOU
13  2006_01_IND_NYG    2006_01_IND_NYG
17   2006_01_SD_OAK     2006_01_LAC_LV
19  2006_01_MIA_PIT    2006_01_MIA_PIT
20  2006_01_MIN_WAS    2006_01_MIN_WAS
24  2006_01_NYJ_TEN    2006_01_NYJ_TEN
32  2006_02_ARI_SEA    2006_02_ARI_SEA
33   2006_02_TB_ATL     2006_02_TB_ATL
34  2006_02_OAK_BAL     2006_02_LV_BAL
35  2006_02_BUF_MIA    2006_02_BUF_MIA


In [21]:
print(
    "Matching games:",
    betting["game_id"].isin(
        team_stats["game_id_normalized"]
    ).sum()
)

print(
    "Unmatched betting games:",
    (~betting["game_id"].isin(
        team_stats["game_id_normalized"]
    )).sum()
)

Matching games: 4927
Unmatched betting games: 0


## 5. Merge betting data with team statistics

In [22]:
team_stats = team_stats.merge(
    betting,
    left_on="game_id_normalized",
    right_on="game_id",
    how="left",
    validate="many_to_one"
)

print("Shape:", team_stats.shape)

Shape: (9854, 68)


In [23]:
print("Rows:", len(team_stats))

print(
    "Unique games:",
    team_stats["game_id_normalized"].nunique()
)

print(
    "Rows per game:"
)
print(
    team_stats
    .groupby("game_id_normalized")
    .size()
    .value_counts()
    .sort_index()
)

print(
    "\nMissing betting data:",
    team_stats["team_favorite_id"].isna().sum()
)

Rows: 9854
Unique games: 4927
Rows per game:
2    4927
Name: count, dtype: int64

Missing betting data: 0


## 6. Normalize favorite identifiers

In [24]:
print(
    team_stats["team_favorite_id"]
    .value_counts()
    .sort_index()
)

team_favorite_id
ARI     248
ATL     318
BAL     416
BUF     292
CAR     258
CHI     246
CIN     312
CLE     204
DAL     390
DEN     332
DET     248
GB      414
HOU     270
IND     352
JAX     184
KC      368
LAC     390
LAR     252
LVR     170
MIA     234
MIN     326
NE      464
NO      394
NYG     250
NYJ     236
PHI     416
PICK     40
PIT     408
SEA     358
SF      332
TB      260
TEN     278
WAS     194
Name: count, dtype: int64


In [25]:
print(
    "Unique favorite IDs:",
    sorted(team_stats["team_favorite_id"].dropna().unique())
)

Unique favorite IDs: ['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAX', 'KC', 'LAC', 'LAR', 'LVR', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'PHI', 'PICK', 'PIT', 'SEA', 'SF', 'TB', 'TEN', 'WAS']


In [26]:
favorite_id_map = {
    "LAR": "LA",
    "LVR": "LV"
}

team_stats["favorite_team"] = (
    team_stats["team_favorite_id"]
    .replace(favorite_id_map)
)

In [27]:
print(
    "Pick'em games:",
    team_stats.loc[
        team_stats["favorite_team"] == "PICK",
        "game_id_normalized"
    ].nunique()
)

Pick'em games: 20


## 7. Create one-row-per-game dataset

In [28]:
games = (
    team_stats[
        [
            "game_id_normalized",
            "season",
            "week",
            "team_favorite_id",
            "favorite_team",
            "score_home",
            "score_away",
            "team_home",
            "team_away",
            "spread_favorite"
        ]
    ]
    .drop_duplicates("game_id_normalized")
    .copy()
)

print("Rows:", len(games))
print("Unique games:", games["game_id_normalized"].nunique())

Rows: 4927
Unique games: 4927


In [29]:
print(
    games[
        [
            "season",
            "week",
            "team_home",
            "team_away",
            "favorite_team",
            "score_home",
            "score_away",
            "spread_favorite"
        ]
    ].head(10)
)

    season  week team_home team_away favorite_team  score_home  score_away  \
0     2006     1       ARI        SF           ARI          34          27   
1     2006     1       CAR       ATL           CAR           6          20   
2     2006     1        TB       BAL            TB           0          27   
3     2006     1        NE       BUF            NE          19          17   
5     2006     1        GB       CHI           CHI           0          26   
6     2006     1        KC       CIN           CIN          10          23   
7     2006     1       CLE        NO           CLE          14          19   
8     2006     1       JAX       DAL           DAL          24          17   
9     2006     1        LA       DEN           DEN          18          10   
10    2006     1       DET       SEA           SEA           6           9   

    spread_favorite  
0              -9.5  
1              -4.5  
2              -3.0  
3             -10.0  
5              -3.5  
6        

### Betting-market upset definition

In [30]:
games["favorite_score"] = games.apply(
    lambda row: (
        row["score_home"]
        if row["favorite_team"] == row["team_home"]
        else row["score_away"]
    ),
    axis=1
)

games["underdog_score"] = games.apply(
    lambda row: (
        row["score_away"]
        if row["favorite_team"] == row["team_home"]
        else row["score_home"]
    ),
    axis=1
)

games["favorite_margin"] = (
    games["favorite_score"] - games["underdog_score"]
)

In [31]:
games["betting_upset"] = (
    (games["favorite_team"] != "PICK") &
    (games["favorite_margin"] < 0)
)

In [32]:
print(
    games[
        [
            "season",
            "week",
            "team_home",
            "team_away",
            "favorite_team",
            "favorite_score",
            "underdog_score",
            "favorite_margin",
            "betting_upset"
        ]
    ].head(15)
)

print("\nBetting-market upsets:", games["betting_upset"].sum())

print(
    "Betting-market upset rate:",
    games["betting_upset"].mean()
)

    season  week team_home team_away favorite_team  favorite_score  \
0     2006     1       ARI        SF           ARI              34   
1     2006     1       CAR       ATL           CAR               6   
2     2006     1        TB       BAL            TB               0   
3     2006     1        NE       BUF            NE              19   
5     2006     1        GB       CHI           CHI              26   
6     2006     1        KC       CIN           CIN              23   
7     2006     1       CLE        NO           CLE              14   
8     2006     1       JAX       DAL           DAL              17   
9     2006     1        LA       DEN           DEN              10   
10    2006     1       DET       SEA           SEA               9   
12    2006     1       HOU       PHI           PHI              24   
13    2006     1       NYG       IND           IND              26   
17    2006     1        LV       LAC           LAC              27   
19    2006     1    

In [33]:
games["upset_margin"] = (
    games["favorite_margin"]
    .clip(upper=0)
    .abs()
)

games.loc[
    games["favorite_team"] == "PICK",
    "upset_margin"
] = 0

In [34]:
print(
    "Upsets:",
    games["betting_upset"].sum()
)

print(
    "Games with nonzero upset margin:",
    (games["upset_margin"] > 0).sum()
)

Upsets: 1637
Games with nonzero upset margin: 1637


## 8. Create reusable team-level game results

In [35]:
home_results = games[
    [
        "season",
        "week",
        "team_home",
        "score_home",
        "score_away"
    ]
].copy()

home_results["wins"] = (
    home_results["score_home"] > home_results["score_away"]
).astype(int)

away_results = games[
    [
        "season",
        "week",
        "team_away",
        "score_away",
        "score_home"
    ]
].copy()

away_results["wins"] = (
    away_results["score_away"] > away_results["score_home"]
).astype(int)

home_results = home_results.rename(
    columns={"team_home": "team"}
)

away_results = away_results.rename(
    columns={"team_away": "team"}
)

In [36]:
team_results = pd.concat(
    [
        home_results[["season", "week", "team", "wins"]],
        away_results[["season", "week", "team", "wins"]]
    ],
    ignore_index=True
)

print("Rows:", len(team_results))
print("Unique teams:", team_results["team"].nunique())

Rows: 9854
Unique teams: 32


In [37]:
# Persist the prepared datasets for downstream notebooks.
from pathlib import Path

data_dir = Path("../data")
data_dir.mkdir(exist_ok=True)

games.to_parquet(data_dir / "games.parquet", index=False)
team_stats.to_parquet(data_dir / "team_stats.parquet", index=False)
team_results.to_parquet(data_dir / "team_results.parquet", index=False)

print("Saved:")
print(" - games.parquet")
print(" - team_stats.parquet")
print(" - team_results.parquet")


Saved:
 - games.parquet
 - team_stats.parquet
 - team_results.parquet


## 9. Final data-quality checks

In [38]:
print("games shape:", games.shape)
print("team_stats shape:", team_stats.shape)
print("team_results shape:", team_results.shape)

print("\nUnique games:", games["game_id_normalized"].nunique())
print("Rows per game in team_stats:")
print(team_stats.groupby("game_id_normalized").size().value_counts().sort_index())

print("\nMissing betting rows:", team_stats["schedule_season"].isna().sum())
print("Missing normalized game IDs:", team_stats["game_id_normalized"].isna().sum())


games shape: (4927, 15)
team_stats shape: (9854, 69)
team_results shape: (9854, 4)

Unique games: 4927
Rows per game in team_stats:
2    4927
Name: count, dtype: int64

Missing betting rows: 0
Missing normalized game IDs: 0
